# Copy-Paste Augmentation 구축 및 검증

이 Notebook은 추가 TS 데이터에서 투명 알약 객체 Pool을 만들고, 실제 Train 배경에 3개 또는 4개 객체를 합성해 학습용 이미지와 Annotation을 구성한 과정을 기록한다.

긴 구현 코드는 반복하지 않고 **입력 snapshot → 객체 추출 → 합성 → Annotation 생성 → 검증 → Dataset 호환**의 재현 가능한 흐름과 실제 완료 결과를 중심으로 정리한다. 원본 이미지·JSON·기존 Crop은 수정하지 않는다.


## 1. 목적과 데이터 보호 원칙

- 최종 선별된 이미지에 대해서만 Copy-Paste 객체를 생성한다.
- 시각화 테두리나 label이 있는 기존 Crop을 보정해서 사용하지 않는다.
- 원본 이미지와 동일 basename의 JSON을 매칭하고, JSON bbox로 원본 픽셀을 다시 Crop한다.
- `idx`, `dl_idx`는 클래스 식별에 사용하지 않으며 `category_id`와 실제 약품명을 사용한다.
- `2483 뮤테란캡슐 100mg`은 제조사 차이로 객체 Pool에서 제외한다.
- `12247 아빌리파이정 10mg`은 기존 JSON bbox를 사용하지 않고 원본 전체에서 객체를 다시 검출하며, 불확실한 결과는 review로 분리한다.
- 기존 객체 PNG와 manifest는 incremental 처리에서 수정·삭제·재처리하지 않는다.


## 2. 전체 처리 흐름

```mermaid
flowchart LR
    A["최종 선별 이미지"] --> B["원본 이미지 + 동일 basename JSON 매칭"]
    B --> C{"category_id 규칙"}
    C -->|"2483"| X["제외"]
    C -->|"12247"| D["원본 전체에서 객체 재검출"]
    C -->|"기타"| E["JSON bbox 재-Crop"]
    D --> F["U2Net/rembg"]
    E --> F
    F --> G["largest component + alpha 후처리"]
    G --> H["투명 PNG 객체 Pool"]
    H --> I["실제 Train 배경 template 선택"]
    I --> J["3알/4알 비중첩 배치"]
    J --> K["최종 위치 기준 새 bbox 계산"]
    K --> L["이미지와 동일 basename JSON 생성"]
    L --> M["1:1·bbox·ID·category 전수 검증"]
```


## 3. 환경별 경로 설정

개인 절대경로를 Notebook에 고정하지 않는다. 프로젝트 외부 대용량 데이터 경로는 환경변수 또는 아래 설정 셀에서 지정한다.


In [ ]:
from pathlib import Path
import csv
import json
import os

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_ROOT = Path(os.environ.get("PILL_DATA_ROOT", PROJECT_ROOT / "data"))

OBJECT_POOL = Path(os.environ.get("COPY_PASTE_OBJECT_POOL", DATA_ROOT / "copy_paste_objects_20260814"))
SAMPLE_V1 = Path(os.environ.get("COPY_PASTE_SAMPLE_V1", DATA_ROOT / "copy_paste_augmented_sample_100"))
SAMPLE_V2 = Path(os.environ.get("COPY_PASTE_SAMPLE_V2", DATA_ROOT / "copy_paste_augmented_sample_100_v2"))

DATASET_MODULE = PROJECT_ROOT / "src" / "PillDetectionDataset.py"


## 4. 객체 Pool 1차 생성

### 입력과 처리

1. `training_selection.xlsx`와 실제 `train_images`를 대조해 최종 유효 선별 이미지를 확정했다.
2. 원본 이미지와 동일 basename JSON을 연결하고 bbox 및 `category_id`를 검증했다.
3. 일반 클래스는 JSON bbox로 원본을 다시 Crop했다.
4. 아빌리파이는 원본 전체 이미지에서 객체를 다시 검출했다.
5. U2Net/rembg와 alpha matting을 적용한 뒤 가장 큰 foreground component를 유지했다.
6. alpha mask가 비어 있거나 여러 조각으로 분리되거나 경계에 닿는 결과를 검증했다.

### 1차 결과

| 항목 | 결과 |
|---|---:|
| 처리 대상 snapshot | 5,062 |
| 정상 투명 PNG | 5,059 |
| 아빌리파이 정상 처리 | 97 |
| 아빌리파이 review | 3 |
| 처리 클래스 | 55 |
| alpha mask 이상 | 0 |

정상 객체는 `category_id_약품명/파일명.png` 구조로 저장하고 `object_manifest.csv`에 원본 이미지, JSON, 처리 방법, mask 면적과 출력 크기를 기록했다.


In [ ]:
# 객체 Pool manifest 기본 무결성 점검
manifest_path = OBJECT_POOL / "object_manifest.csv"

if manifest_path.is_file():
    with manifest_path.open(encoding="utf-8-sig", newline="") as file:
        object_rows = list(csv.DictReader(file))

    object_keys = {(row["category_id"], row["file_name"]) for row in object_rows}
    missing_outputs = [row["output_path"] for row in object_rows if not Path(row["output_path"]).is_file()]
    print({
        "manifest_rows": len(object_rows),
        "unique_category_file_keys": len(object_keys),
        "missing_outputs": len(missing_outputs),
    })
else:
    print("OBJECT_POOL 경로를 설정하면 manifest를 점검할 수 있습니다.")


## 5. 객체 Pool 2차 incremental 업데이트

추가 TS 다운로드 완료 후 기존 `object_manifest.csv`와 최신 최종 선별 목록을 `(category_id, file_name)`으로 비교했다. 기존 5,059개 객체는 재처리하지 않고 신규 TS 405장만 동일한 파이프라인으로 처리했다.

| category_id | 약품명 | 기존 | 신규 | 최종 |
|---:|---|---:|---:|---:|
| 16232 | 리피토정 20mg | 21 | 79 | 100 |
| 16262 | 크레스토정 20mg | 23 | 77 | 100 |
| 32310 | 글리아타민연질캡슐 | 14 | 86 | 100 |
| 36637 | 로수젯정10/5밀리그램 | 19 | 81 | 100 |
| 38162 | 로수바미브정 10/20mg | 18 | 82 | 100 |

최종 객체 Pool은 **5,464장**, manifest도 **5,464행**이며 `(category_id, file_name)` 중복과 파일 누락은 0건이다. 기존 manifest 백업과 신규 405행 manifest를 별도로 남겨 incremental 변경 범위를 추적할 수 있게 했다.


## 6. 실제 Train 배경 활용

임의의 흰색·검정색·파란색 단색을 만들지 않고, 기존 Train의 실제 연회색 촬영 배경을 사용했다.

### 1차 방식과 발견 사항

- 100개의 서로 다른 Train 이미지에서 bbox 바깥의 실제 배경 band를 사용했다.
- 100장 중 4장(`000012`, `000022`, `000044`, `000093`)에서 원본 Annotation에 포함되지 않은 알약 흔적이 배경에 남아 있었다.
- 붙여 넣은 객체의 bbox는 정상이지만 미표기 객체로 학습될 수 있으므로 4장은 warning 대상으로 판정했다.

### 2차 개선 방식

- Train 이미지의 모든 알려진 bbox와 겹치지 않는 220×220 영역만 후보로 사용했다.
- 가장자리 광택 영역을 제외하고 밝기 표준편차·gradient·채도 변화가 가장 낮은 실제 배경 patch를 선택했다.
- 선택 patch를 합성 canvas로 확장했으며, 100개의 서로 다른 Train 배경 source를 사용했다.
- 2차 전수 육안 검증에서는 미표기 알약 잔상이 발견되지 않았다.


## 7. 3알/4알 Copy-Paste 합성

- 3알 이미지: 자연스러운 삼각형 배치를 기준으로 좌표를 무작위 변형한다.
- 4알 이미지: 상·하·좌·우로 분산하되 고정 grid가 되지 않도록 좌표를 변형한다.
- 객체별 크기와 회전을 제한 범위에서 변형한다.
- 객체 간 겹침, 이미지 경계 초과, 지나치게 좁은 간격을 금지한다.
- 투명 PNG의 alpha channel을 사용해 실제 Train 배경에 합성한다.

학습용 `images/`에는 bbox 선이나 category label을 그리지 않는다. 시각화 결과는 별도의 `review/`와 `bbox_visual_validation/`에만 저장한다.


## 8. 새 bbox와 Annotation 생성

원본 JSON bbox를 복사하지 않고, 변형된 투명 PNG가 최종 canvas에 배치된 실제 위치와 alpha mask 범위로 새 bbox `[x, y, width, height]`를 계산했다.

각 합성 JSON은 다음 구조를 유지한다.

- `images`: 동일 basename 이미지, 976×1280, 새 `image_id`, 실제 배경 source 추적 정보
- `annotations`: 모든 3개 또는 4개 객체의 새 bbox, 원래 `category_id`, 새 `annotation_id`, source 객체 파일명
- `categories`: 각 annotation의 `category_id`와 실제 약품명

이미지와 JSON은 확장자만 다른 동일 basename이며 ID는 1차/2차 및 기존 데이터와 충돌하지 않는 범위로 생성했다.


In [ ]:
def validate_copy_paste_folder(root: Path):
    images = {path.stem: path for path in (root / "images").glob("*.png")}
    annotations = {path.stem: path for path in (root / "annotations").glob("*.json")}
    errors, image_ids, annotation_ids = [], [], []

    for stem, image_path in sorted(images.items()):
        if stem not in annotations:
            errors.append((stem, "missing_json"))
            continue
        payload = json.loads(annotations[stem].read_text(encoding="utf-8"))
        image_record = payload["images"][0]
        objects = payload["annotations"]
        image_ids.append(int(image_record["id"]))

        if image_record["file_name"] != image_path.name:
            errors.append((stem, "file_name_mismatch"))
        if len(objects) not in (3, 4):
            errors.append((stem, f"unexpected_object_count={len(objects)}"))

        for annotation in objects:
            x, y, width, height = annotation["bbox"]
            annotation_ids.append(int(annotation["id"]))
            if width <= 0 or height <= 0 or x < 0 or y < 0:
                errors.append((stem, "invalid_bbox"))
            if x + width > 976 or y + height > 1280:
                errors.append((stem, "bbox_out_of_bounds"))

    return {
        "images": len(images),
        "jsons": len(annotations),
        "basename_1to1": len(set(images) & set(annotations)),
        "unique_image_ids": len(set(image_ids)),
        "unique_annotation_ids": len(set(annotation_ids)),
        "errors": errors,
    }

# validate_copy_paste_folder(SAMPLE_V1)
# validate_copy_paste_folder(SAMPLE_V2)


## 9. 1차 Copy-Paste 결과

| 검증 항목 | 결과 |
|---|---:|
| 합성 이미지 / JSON | 100 / 100 |
| 3알 / 4알 이미지 | 50 / 50 |
| 전체 객체 | 350 |
| 대상 소수 클래스 | 28 |
| 이미지 ↔ JSON 1:1 | 100 / 100 |
| bbox–alpha 위치 검증 | 100 / 100 |
| image_id / annotation_id 중복 | 0 / 0 |
| 구조·ID 검증 오류 | 0 |
| 배경 미표기 객체 warning | 4 |

1차는 소수 클래스별 등장 횟수를 12~13회로 균형 배분했다. 배경 warning 4장은 bbox 오류가 아니라 background template 오염 문제이므로 별도 검수 또는 제외 대상으로 관리한다.


## 10. 2차 Copy-Paste 결과

2차는 incremental로 추가된 다섯 클래스의 신규 객체만 사용해 독립된 테스트 세트를 만들었다.

| 검증 항목 | 결과 |
|---|---:|
| 합성 이미지 / JSON | 100 / 100 |
| 파일명 범위 | `copy_paste_000101`~`copy_paste_000200` |
| 3알 / 4알 이미지 | 50 / 50 |
| 전체 객체 | 350 |
| 클래스 | 5 |
| 클래스별 등장 횟수 | 각 70 |
| 실제 Train 배경 source | 100 |
| 이미지 ↔ JSON 1:1 | 100 / 100 |
| bbox–alpha 위치 검증 | 100 / 100 |
| bbox 경계 초과 / 객체 겹침 | 0 / 0 |
| image_id / annotation_id 중복 | 0 / 0 |
| 배경 미표기 객체 warning | 0 |

2차 파일명은 1차와 충돌하지 않으면서 Dataset 합성 파일명 규칙을 만족한다.


In [ ]:
# 저장된 검증 summary 확인
for label, folder in (("v1", SAMPLE_V1), ("v2", SAMPLE_V2)):
    summary_path = folder / "validation_summary.json"
    if summary_path.is_file():
        print(label, json.loads(summary_path.read_text(encoding="utf-8")))
    else:
        print(label, "경로를 설정하면 validation_summary.json을 확인할 수 있습니다.")


## 11. PillDetectionDataset 호환

기존 Train 로딩 경로는 유지하면서 합성 데이터만 별도 분기하도록 `src/PillDetectionDataset.py`를 최소 확장했다.

### 기존 Train 경로

- 기존 `K-` 6자리 약품 코드 3~4개 파일명 정규식을 그대로 사용한다.
- `combination_key_json/K-{category_id}/동일_basename.json` 탐색을 유지한다.
- 객체별 JSON의 첫 annotation을 읽는 기존 동작을 유지한다.
- 파일명 약품 코드와 JSON `drug_N` 검증을 유지한다.

### 합성 데이터 경로

- `copy_paste_숫자.png`만 합성 파일명으로 인식한다.
- 평면 `annotations/동일_basename.json`을 찾는다.
- `annotations[]` 전체 3~4개를 모두 로드한다.
- 각 annotation의 `category_id`와 동일한 `categories[].id`를 연결한다.
- 새 bbox, `category_id`, `image_id`, `annotation_id`를 보존한다.
- 촬영각도와 원본 조합 파일명 검증은 적용하지 않는다.

두 경로 모두 동일한 target 키(`boxes`, `labels`, `image_id`, `area`, `iscrowd`, `annotation_id`, `ignore`)를 반환한다.


In [ ]:
# PyTorch가 설치된 학습 환경에서 실행하는 smoke test 예시
# from src.PillDetectionDataset import PillDetectionDataset
#
# synthetic_dataset = PillDetectionDataset(
#     SAMPLE_V2,
#     image_dir_name="images",
#     annotation_dir_name="annotations",
#     validate_image_size=True,
# )
# image, target, metadata = synthetic_dataset[0]
# print(metadata["file_name"], target["boxes"].shape, metadata["num_pills"])


## 12. 실제 Dataset 로딩 검증

Dataset 로딩 로직으로 다음 샘플을 확인했다.

| 구분 | 파일 | bbox 수 | category_id |
|---|---|---:|---|
| 기존 Train | `K-003483-025367-027733-035206...png` | 4 | 3483, 25367, 27733, 35206 |
| Copy-Paste 3객체 | `copy_paste_000001.png` | 3 | 3544, 28763, 31885 |
| Copy-Paste 4객체 | `copy_paste_000051.png` | 4 | 13395, 22362, 3544, 4543 |

합성 100장 전체를 Dataset 파싱 결과와 JSON 원본으로 대조했으며 bbox 개수, category ID, annotation ID 불일치는 0건이었다. 실제 학습은 이 검증 과정에서 실행하지 않았다.


## 13. 최종 체크리스트

1. 객체 Pool manifest의 `(category_id, file_name)` 중복이 0건인가?
2. manifest의 모든 `output_path`가 실제 RGBA PNG로 존재하는가?
3. 합성 이미지와 JSON basename이 1:1인가?
4. 모든 이미지의 객체 수가 3개 또는 4개인가?
5. bbox가 alpha mask의 실제 합성 위치와 일치하는가?
6. bbox가 이미지 경계를 벗어나거나 객체끼리 겹치지 않는가?
7. `category_id`, `image_id`, `annotation_id`가 유효하고 중복되지 않는가?
8. 학습용 이미지에 bbox line이나 label이 없는가?
9. 실제 Train 배경에 미표기 알약이나 과도한 잔상이 없는가?
10. 기존 Train Dataset 로딩 경로의 regression이 없는가?

2차 결과는 위 항목을 모두 통과했다. 1차 결과는 배경 warning 4장을 학습 사용 전에 제외하거나 깨끗한 배경으로 다시 합성해야 한다.


## 14. 최종 공유 데이터 사용 (Colab + Google Drive)

최종 Copy-Paste v1/v2 데이터는 팀 Google Drive 공유폴더를 정본으로 사용한다. 이미지는 ZIP으로, Annotation은 버전별 폴더로 보관되어 있다. 아래 셀은 개인 Mac 경로를 사용하지 않으며, Drive 마운트 후 파일 수와 PNG↔JSON basename 1:1 대응을 확인한다. 공유폴더를 `내 드라이브`에 바로가기 추가한 위치가 다르면 `SHARED_ROOT`만 수정한다.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# 공유폴더 바로가기가 MyDrive 아래에 있는 기본 예시입니다.
+# 팀 Drive를 사용하는 경우에도 /content/drive 아래의 실제 위치로 이 값만 바꾸세요.
+SHARED_ROOT = Path('/content/drive/MyDrive/코드잇_파트2_3팀_프로젝트/추가데이터셋추출')
+
+V1_IMAGE_ZIP = SHARED_ROOT / 'copy_paste_augmented_sample_100.zip'
+V2_IMAGE_ZIP = SHARED_ROOT / 'copy_paste_augmented_sample_100_v2.zip'
+V1_ANNOTATION_DIR = SHARED_ROOT / 'copy_paste_augmented_sample100_v1_annotations'
+V2_ANNOTATION_DIR = SHARED_ROOT / 'copy_paste_augmented_sample100_v2_annotations'
+
+for path in (V1_IMAGE_ZIP, V2_IMAGE_ZIP, V1_ANNOTATION_DIR, V2_ANNOTATION_DIR):
+    print(f'{path}: {"OK" if path.exists() else "NOT FOUND"}')
+

In [ ]:
from collections import Counter
+from zipfile import ZipFile
+
+def png_basenames_from_zip(zip_path):
+    with ZipFile(zip_path) as archive:
+        names = [Path(name).stem for name in archive.namelist() if Path(name).suffix.lower() == '.png']
+    return names
+
+def verify_version(version, image_zip, annotation_dir):
+    png_names = png_basenames_from_zip(image_zip)
+    json_names = [path.stem for path in annotation_dir.glob('*.json')]
+    png_counts, json_counts = Counter(png_names), Counter(json_names)
+    duplicate_png = sorted(name for name, count in png_counts.items() if count > 1)
+    duplicate_json = sorted(name for name, count in json_counts.items() if count > 1)
+    missing_json = sorted(set(png_names) - set(json_names))
+    missing_png = sorted(set(json_names) - set(png_names))
+    matched = len(set(png_names) & set(json_names))
+    print(f'[{version}] PNG={len(png_names)}, JSON={len(json_names)}, matched={matched}')
+    print(f'  duplicate PNG basename={len(duplicate_png)}, duplicate JSON basename={len(duplicate_json)}')
+    print(f'  PNG without JSON={len(missing_json)}, JSON without PNG={len(missing_png)}')
+    assert not duplicate_png and not duplicate_json, f'{version}: duplicate basename detected'
+    assert not missing_json and not missing_png, f'{version}: PNG/JSON basename mismatch'
+    assert len(png_names) == len(json_names), f'{version}: file count mismatch'
+    return {'version': version, 'images': len(png_names), 'annotations': len(json_names), 'matched': matched}
+
+final_data_check = [
+    verify_version('v1', V1_IMAGE_ZIP, V1_ANNOTATION_DIR),
+    verify_version('v2', V2_IMAGE_ZIP, V2_ANNOTATION_DIR),
+]
+final_data_check
+